# Bronze to silver - Cleaning & transforming dimension tables

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import DateType, IntegerType
catalog_name = 'edutrack'

## Students
**Issues:** department casing anomalies, city casing anomalies, NULL phone, date_of_birth as string

In [0]:
df = spark.table('edutrack.bronze.brz_students')
print('Row count:', df.count())
df.select('department').distinct().orderBy('department').show(truncate=False)

In [0]:
dept_fixes = {'computer science':'Computer Science','ELECTRONICS':'Electronics','Mech':'Mechanical'}
df_silver = df.replace(to_replace=dept_fixes, subset=['department'])
df_silver = df_silver.withColumn('city', F.initcap(F.col('city')))
df_silver = df_silver.fillna('Not Available', subset=['phone'])
df_silver = df_silver.withColumn('date_of_birth', F.to_date(F.col('date_of_birth'),'yyyy-MM-dd'))
df_silver.select('department','city','phone').show(10)

In [0]:
df_silver.write.format('delta').mode('overwrite').option('mergeSchema','true').saveAsTable('edutrack.silver.slv_students')
print('slv_students:', spark.table('edutrack.silver.slv_students').count(),'rows')

## Courses - clean as is, promoting directly

In [0]:
df = spark.table('edutrack.bronze.brz_courses')
df.write.format('delta').mode('overwrite').option('mergeSchema','true').saveAsTable('edutrack.silver.slv_courses')
print('slv_courses:', spark.table('edutrack.silver.slv_courses').count(),'rows')

## Instructors - clean as is, promoting directly

In [0]:
df = spark.table('edutrack.bronze.brz_instructors')
df.write.format('delta').mode('overwrite').option('mergeSchema','true').saveAsTable('edutrack.silver.slv_instructors')
print('slv_instructors:', spark.table('edutrack.silver.slv_instructors').count(),'rows')

## Date Dimension
**Issues:** date string to DateType, duplicates, day_name casing, negative week_of_year, add date_id key

In [0]:
df = spark.table('edutrack.bronze.brz_date')
df_silver = df.withColumn('date', F.to_date(F.col('date'),'dd-MM-yyyy'))
print('Duplicates:', df_silver.groupBy('date').count().filter(F.col('count')>1).count())
df_silver = df_silver.dropDuplicates(['date'])
df_silver = df_silver.withColumn('day_name', F.initcap(F.col('day_name')))
df_silver = df_silver.withColumn('week_of_year', F.abs(F.col('week_of_year')))
df_silver = df_silver.withColumn('quarter', F.concat(F.lit('Q'), F.col('quarter'), F.lit('-'), F.col('year')))
df_silver = df_silver.withColumn('date_id', F.date_format(F.col('date'),'yyyyMMdd').cast(IntegerType()))
df_silver.show(5)

In [0]:
df_silver.write.format('delta').mode('overwrite').option('mergeSchema','true').saveAsTable('edutrack.silver.slv_date')
print('slv_date:', spark.table('edutrack.silver.slv_date').count(),'rows')